# Ad Topic Taxonomy from the Google Topics API

Builds the list of ad categories used to generate the synthetic ad inventory.

Source: the [Topics API taxonomy v2](https://github.com/patcg-individual-drafts/topics/blob/main/taxonomy_v2.md),
a public, hierarchical list of interest topics (e.g. `/Autos & Vehicles/Motor Vehicles (By Type)/Trucks`)
maintained for Chrome's Privacy Sandbox. It is advertiser-oriented, which makes it a good basis
for ad categories.

**Output:** `data/processed/ad_topics/taxonomy_subtopics.csv` with columns `Topic_Level_1`, `sub_topic`.
It is read by `src/ad_generation/scripts/generate_bulk_ads.py` (`GenerateBulkAdsService`) via
`TAXONOMY_CSV_PATH` in `src/ad_generation/config.py`. If you change the output path or columns, update that config too.

| Step | What happens |
|---|---|
| 1 | Download the Markdown table and parse it into a DataFrame |
| 2 | Clean `ID` / `Topic` columns |
| 3 | Split the `/`-delimited path into `Topic_Level_1..N` |
| 4 | Merge levels 2+ into one `sub_topic` string |
| 5 | Drop top-level categories that don't suit product ads |
| 6 | Export to CSV |

## 1. Setup

In [1]:
import os
import pandas as pd

## 2. Load the taxonomy

The taxonomy is published as a Markdown table, so it can be read straight off GitHub with
`read_csv(sep="|")`. That parse leaves two artifacts to clean up:

- column names are padded with spaces
- the first data row is the `|---|---|` separator line

In [2]:
# Raw Markdown table: | ID | Topic |
URL = "https://raw.githubusercontent.com/patcg-individual-drafts/topics/main/taxonomy_v2.md"

df = pd.read_csv(
    URL,
    sep="|",
    skipinitialspace=True
)

# Clean Markdown table formatting
df.columns = [c.strip() for c in df.columns]  # strip padding from column names
df = df.iloc[1:, :]  # drop the |---|---| separator row

In [3]:
# Raw parse still has empty edge columns (from the leading/trailing '|') and padded values
df.head()

,Unnamed: 0,ID,Topic,Unnamed: 3
1,NaN,1,/Arts & Entertainment ...,NaN
2,NaN,350,/Arts & Entertainment/Celebrities & Entertainm...,NaN
3,NaN,351,/Arts & Entertainment/Comics & Animation ...,NaN
4,NaN,352,/Arts & Entertainment/Events & Listings ...,NaN
5,NaN,353,"/Arts & Entertainment/Events & Listings/Bars, ...",NaN


## 3. Clean `ID` and `Topic`

Strip whitespace, cast `ID` to int, and keep only the two useful columns
(this also drops the empty columns created by the table's outer `|` borders).

In [4]:
# Topic id: padded string -> int
df["ID"] = (
    df["ID"]
    .astype(str)
    .str.strip()
    .astype(int)
)

# Topic path, e.g. "/Arts & Entertainment/Music & Audio"
df["Topic"] = (
    df["Topic"]
    .astype(str)
    .str.strip()
)

df = df[['ID', 'Topic']]

print(df.head())
print(f"Total topics: {len(df)}")

    ID                                              Topic
1    1                              /Arts & Entertainment
2  350  /Arts & Entertainment/Celebrities & Entertainm...
3  351           /Arts & Entertainment/Comics & Animation
4  352            /Arts & Entertainment/Events & Listings
5  353  /Arts & Entertainment/Events & Listings/Bars, ...
Total topics: 469


## 4. Split the topic hierarchy

Each `Topic` is a path such as `/Autos & Vehicles/Motor Vehicles (By Type)/Trucks`.
Splitting on `/` gives one column per level: `Topic_Level_1` (the top-level category),
`Topic_Level_2`, and so on. Shallower paths get `None` in the deeper levels.

In [5]:
# Separate Topic by delimiter '/'
topic_expanded = df['Topic'].str.split('/', expand=True)
# The first column is empty because of the leading slash
topic_expanded = topic_expanded.iloc[:, 1:]
topic_expanded.columns = [f'Topic_Level_{i+1}' for i in range(topic_expanded.shape[1])]
df = pd.concat([df, topic_expanded], axis=1)
df.head()

,ID,Topic,Topic_Level_1,Topic_Level_2,Topic_Level_3,Topic_Level_4,Topic_Level_5
1,1,/Arts & Entertainment,Arts & Entertainment,NaN,NaN,NaN,NaN
2,350,/Arts & Entertainment/Celebrities & Entertainm...,Arts & Entertainment,Celebrities & Entertainment News,NaN,NaN,NaN
3,351,/Arts & Entertainment/Comics & Animation,Arts & Entertainment,Comics & Animation,NaN,NaN,NaN
4,352,/Arts & Entertainment/Events & Listings,Arts & Entertainment,Events & Listings,NaN,NaN,NaN
5,353,"/Arts & Entertainment/Events & Listings/Bars, ...",Arts & Entertainment,Events & Listings,"Bars, Clubs & Nightlife",NaN,NaN


## 5. Build `sub_topic`

Ad generation works on a (category, sub-topic) pair, so levels 2 through 5 are joined with `" + "`
into a single `sub_topic` string, e.g. `Motor Vehicles (By Type) + Trucks`.

Top-level-only rows (e.g. `/Arts & Entertainment`) have no level 2, so their `sub_topic` is `None`.

In [6]:
# Merge Topic_Level_2 through Topic_Level_5 into sub_topic
df['sub_topic'] = df['Topic_Level_2'].copy()
for i in range(3, 6):
    col = f'Topic_Level_{i}'
    if col in df.columns:  # guard: taxonomy depth may change between versions
        mask = df[col].notna() & (df[col] != '')
        df.loc[mask, 'sub_topic'] = df.loc[mask, 'sub_topic'] + ' + ' + df.loc[mask, col]

df[['Topic', 'sub_topic']].head(10)

,Topic,sub_topic
1,/Arts & Entertainment,NaN
2,/Arts & Entertainment/Celebrities & Entertainm...,Celebrities & Entertainment News
3,/Arts & Entertainment/Comics & Animation,Comics & Animation
4,/Arts & Entertainment/Events & Listings,Events & Listings
5,"/Arts & Entertainment/Events & Listings/Bars, ...","Events & Listings + Bars, Clubs & Nightlife"
6,/Arts & Entertainment/Events & Listings/Concer...,Events & Listings + Concerts & Music Festivals
7,/Arts & Entertainment/Events & Listings/Event ...,Events & Listings + Event Ticket Sales
8,/Arts & Entertainment/Events & Listings/Expos ...,Events & Listings + Expos & Conventions
9,/Arts & Entertainment/Events & Listings/Film F...,Events & Listings + Film Festivals
10,/Arts & Entertainment/Events & Listings/Food &...,Events & Listings + Food & Beverage Events


## 6. Drop categories unsuited to ads

Some top-level categories are mostly informational or social (news, law, communities, etc.)
and don't map well to products or services someone would advertise, so they are removed.
Edit `topics_to_remove` to change the ad inventory's coverage.

In [7]:
# Remove rows where Topic_Level_1 matches specific categories
topics_to_remove = [
    "Books & Literature",
    "Hobbies & Leisure",
    "People & Society",
    "Pets & Animals",
    "News",
    "Law & Government",
    "Online Communities"
]

df = df[~df['Topic_Level_1'].isin(topics_to_remove)]
print(f"Remaining topics: {len(df)}")
df['Topic_Level_1'].value_counts()  # sub-topics per remaining top-level category

Remaining topics: 432


Topic_Level_1
Home & Garden              60
Arts & Entertainment       49
Shopping                   45
Computers & Electronics    35
Autos & Vehicles           34
Jobs & Education           34
Sports                     29
Beauty & Fitness           28
Finance                    22
Business & Industrial      21
Food & Drink               19
Travel & Transportation    18
Internet & Telecom         16
Games                      13
Real Estate                 9
Name: count, dtype: int64

## 7. Export

Writes `Topic_Level_1` + `sub_topic` for the bulk ad generator. The path is relative to this
notebook (`analysis/ads_domain/new/`) and resolves to `data/processed/ad_topics/` at the repo root.

In [8]:
# Export taxonomy for bulk ad generation (consumed via TAXONOMY_CSV_PATH in src/ad_generation/config.py)
out_dir = '../../../data/processed/ad_topics'
os.makedirs(out_dir, exist_ok=True)
df[['Topic_Level_1', 'sub_topic']].to_csv(os.path.join(out_dir, 'taxonomy_subtopics.csv'), index=False)
print('Saved taxonomy_subtopics.csv')

Saved taxonomy_subtopics.csv
